In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_4_day_1.csv',
    'prices_round_4_day_2.csv',
    'prices_round_4_day_3.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Configuration ──────────────────────────────────────────────────────────────
POSITION_LIMIT = 200        # max abs position
HALF_SPREAD    = 10      # cost per side per unit (adjust to your product)
products       = ["HYDROGEL_PACK"]
days           = [1, 2, 3]
# ───────────────────────────────────────────────────────────────────────────────


def oracle_dp(prices: np.ndarray, half_spread: float, limit: int):
    """
    Backward-induction DP to find the globally optimal trade sequence.

    State : (timestep t, position j) where j ∈ [-L, L] stored offset by L
    Value : maximum *cash* achievable from timestep t onward,
            NOT including mark-to-market on the open position
            (that is added at backtrack / terminal liquidation).

    At each step we may trade any integer Δ ∈ [-L-pos, L-pos],
    crossing the spread on every share transacted.

    Returns
    -------
    trades   : list[int]  — signed trade size at each timestep (0 = hold)
    max_pnl  : float      — realised PnL (with terminal flat liquidation)
    pnl_path : np.ndarray — cumulative PnL at each step after backtracking
    """
    n     = len(prices)
    L     = limit
    n_pos = 2 * L + 1          # index 0 … 2L  ↔  position -L … +L

    NEG_INF = -np.inf
    dp      = np.full((n, n_pos), NEG_INF)
    par     = np.zeros((n, n_pos), dtype=np.int16)   # parent delta

    dp[0][L] = 0.0             # start flat, zero cash

    for t in range(1, n):
        p = prices[t]
        hs = half_spread
        for j in range(n_pos):
            pos = j - L
            best_val  = NEG_INF
            best_delta = 0
            # Try every valid trade size from the PREVIOUS state
            for delta in range(-L - pos, L - pos + 1):
                prev_j = j - delta          # where we were before trading
                if prev_j < 0 or prev_j >= n_pos:
                    continue
                prev_val = dp[t-1][prev_j]
                if prev_val == NEG_INF:
                    continue
                # Cash change: pay ask (p+hs) per unit bought, receive bid (p-hs) per unit sold
                if delta > 0:
                    cash = -delta * (p + hs)
                elif delta < 0:
                    cash = -delta * (p - hs)   # delta<0 → -delta>0, net positive cash
                else:
                    cash = 0.0
                val = prev_val + cash
                if val > best_val:
                    best_val   = val
                    best_delta = delta
            dp[t][j]  = best_val
            par[t][j] = best_delta

    # Terminal: liquidate remaining position at mid (no extra spread for simplicity,
    # or use (p-hs)*pos if you want to penalise the final unwind)
    terminal = np.full(n_pos, NEG_INF)
    p_last   = prices[-1]
    for j in range(n_pos):
        if dp[n-1][j] == NEG_INF:
            continue
        pos = j - L
        # liquidate: sell pos shares at (p_last - hs) if long, buy -pos at (p_last + hs) if short
        liq = pos * (p_last - hs) if pos > 0 else -pos * -(p_last + hs) if pos < 0 else 0
        terminal[j] = dp[n-1][j] + liq

    # Backtrack
    best_end = int(np.argmax(terminal))
    max_pnl  = terminal[best_end]

    trades_rev = []
    j = best_end
    for t in range(n - 1, 0, -1):
        d = int(par[t][j])
        trades_rev.append(d)
        j -= d
    trades_rev.append(0)          # t=0: no incoming trade
    trades = trades_rev[::-1]

    # Reconstruct cumulative PnL step-by-step for the chart
    position   = 0
    cash       = 0.0
    pnl_path   = np.zeros(n)
    for t in range(n):
        d = trades[t]
        if d > 0:
            cash -= d * (prices[t] + half_spread)
        elif d < 0:
            cash += (-d) * (prices[t] - half_spread)
        position += d
        pnl_path[t] = cash + position * prices[t]   # mark-to-market PnL

    return trades, max_pnl, pnl_path


# ── Per-product, per-day analysis & visualisation ─────────────────────────────
for day in days:
    for product in products:
        subset = df_total[
            (df_total['product'] == product) & (df_total['day'] == day)
        ].copy().sort_values('timestamp').reset_index(drop=True)

        if subset.empty:
            print(f"No data for {product} day {day}")
            continue

        subset['mid_price'] = subset['mid_price'].replace(0, np.nan).ffill()
        prices_arr = subset['mid_price'].to_numpy(dtype=float)
        timestamps = subset['timestamp'].to_numpy()

        trades, max_pnl, pnl_path = oracle_dp(prices_arr, HALF_SPREAD, POSITION_LIMIT)
        trades_arr = np.array(trades)

        # ── Classify each timestep ────────────────────────────────────────────
        buy_mask  = trades_arr > 0
        sell_mask = trades_arr < 0

        # For annotation: scale marker size to trade size
        buy_sizes  = np.clip(np.abs(trades_arr[buy_mask])  * 4, 8, 24)
        sell_sizes = np.clip(np.abs(trades_arr[sell_mask]) * 4, 8, 24)

        # ── Figure ────────────────────────────────────────────────────────────
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.06,
            row_heights=[0.65, 0.35],
            subplot_titles=(
                f"Oracle-Optimal Trades — {product} (Day {day})",
                f"Cumulative PnL  [oracle max = {max_pnl:,.1f}]"
            )
        )

        # Price path (background)
        fig.add_trace(go.Scatter(
            x=timestamps, y=prices_arr,
            mode='lines', name='Mid Price',
            line=dict(color='rgba(120,120,120,0.45)', width=1.2)
        ), row=1, col=1)

        # Buy markers (triangles up, green)
        if buy_mask.any():
            fig.add_trace(go.Scatter(
                x=timestamps[buy_mask],
                y=prices_arr[buy_mask],
                mode='markers',
                name='Buy',
                marker=dict(
                    symbol='triangle-up',
                    size=buy_sizes,
                    color='#00c97a',
                    line=dict(color='#007a49', width=1)
                ),
                text=[f"+{d}" for d in trades_arr[buy_mask]],
                hovertemplate="BUY %{text}<br>Price: %{y:.2f}<br>t=%{x}<extra></extra>"
            ), row=1, col=1)

        # Sell markers (triangles down, red)
        if sell_mask.any():
            fig.add_trace(go.Scatter(
                x=timestamps[sell_mask],
                y=prices_arr[sell_mask],
                mode='markers',
                name='Sell',
                marker=dict(
                    symbol='triangle-down',
                    size=sell_sizes,
                    color='#ff4455',
                    line=dict(color='#aa1122', width=1)
                ),
                text=[f"{d}" for d in trades_arr[sell_mask]],
                hovertemplate="SELL %{text}<br>Price: %{y:.2f}<br>t=%{x}<extra></extra>"
            ), row=1, col=1)

        # PnL curve (bottom panel)
        pnl_color = [
            '#00c97a' if v >= 0 else '#ff4455'
            for v in pnl_path
        ]
        fig.add_trace(go.Scatter(
            x=timestamps, y=pnl_path,
            mode='lines', name='Cumulative PnL',
            line=dict(color='#4488ff', width=1.5),
            fill='tozeroy',
            fillcolor='rgba(68,136,255,0.12)'
        ), row=2, col=1)

        fig.add_hline(y=0, line_dash='dot', line_color='gray', row=2, col=1)

        fig.update_layout(
            height=750,
            template='plotly_white',
            showlegend=True,
            legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
            title_text=f"DP Oracle — {product} | Day {day} | Limit={POSITION_LIMIT} | HalfSpread={HALF_SPREAD}"
        )
        fig.update_yaxes(title_text='Price',     row=1, col=1)
        fig.update_yaxes(title_text='PnL (shells)', row=2, col=1)
        fig.update_xaxes(title_text='Timestamp', row=2, col=1)

        # Trade summary
        n_buys  = int(buy_mask.sum())
        n_sells = int(sell_mask.sum())
        print(f"{product} Day {day}: {n_buys} buys, {n_sells} sells | Oracle PnL = {max_pnl:,.1f}")

        fig.show()


HYDROGEL_PACK Day 1: 317 buys, 283 sells | Oracle PnL = 293,097.0


HYDROGEL_PACK Day 2: 318 buys, 313 sells | Oracle PnL = 269,776.5


HYDROGEL_PACK Day 3: 271 buys, 291 sells | Oracle PnL = 278,347.5


In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Configuration — must match the DP cell above ──────────────────────────────
POSITION_LIMIT = 200
HALF_SPREAD    = 10
products       = ["HYDROGEL_PACK"]
days           = [1, 2, 3]

# Trade file pattern — adjust path prefix if needed
TRADE_FILE_PATTERN = "trades_round_4_day_{day}.csv"

# Distinct colours per trader (up to ~10 bots)
TRADER_PALETTE = [
    "#f59e0b", "#8b5cf6", "#06b6d4", "#ec4899",
    "#f97316", "#14b8a6", "#6366f1", "#84cc16",
    "#e11d48", "#0ea5e9",
]
# ─────────────────────────────────────────────────────────────────────────────


def load_bot_trades(day: int, product: str) -> pd.DataFrame:
    """Load trades CSV and return signed per-trader events for one product."""
    path = TRADE_FILE_PATTERN.format(day=day)
    try:
        df = pd.read_csv(path, sep=";")
    except FileNotFoundError:
        print(f"  Warning: {path} not found, skipping bot overlay.")
        return pd.DataFrame()

    df = df[df["symbol"] == product].copy()
    if df.empty:
        return df

    # Expand into per-trader signed rows
    rows = []
    for _, row in df.iterrows():
        rows.append(dict(
            timestamp=row["timestamp"],
            trader=row["buyer"],
            qty=+int(row["quantity"]),
            price=row["price"],
        ))
        rows.append(dict(
            timestamp=row["timestamp"],
            trader=row["seller"],
            qty=-int(row["quantity"]),
            price=row["price"],
        ))
    return pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)


def compute_bot_stats(bot_df: pd.DataFrame, price_series: pd.Series, timestamps: np.ndarray):
    """
    For each trader build:
      - position_path : cumulative position interpolated onto price timestamps
      - pnl_path      : mark-to-market PnL on price timestamps
      - buy_ts/prices, sell_ts/prices for scatter markers
    """
    stats = {}
    mid_lookup = dict(zip(timestamps, price_series))

    for trader, grp in bot_df.groupby("trader"):
        grp = grp.sort_values("timestamp")

        # Running position & cash from actual trade prices
        pos, cash = 0, 0.0
        pos_at_ts   = {}   # timestamp → position after trade
        pnl_at_ts   = {}

        buy_ts, buy_px, buy_qty   = [], [], []
        sell_ts, sell_px, sell_qty = [], [], []

        for _, r in grp.iterrows():
            ts = r["timestamp"]
            q  = r["qty"]
            p  = r["price"]
            if q > 0:
                cash -= q * p
                buy_ts.append(ts); buy_px.append(p); buy_qty.append(q)
            else:
                cash += (-q) * p
                sell_ts.append(ts); sell_px.append(p); sell_qty.append(-q)
            pos += q
            mid = mid_lookup.get(ts, p)
            pos_at_ts[ts] = pos
            pnl_at_ts[ts] = cash + pos * mid

        # Interpolate position & PnL onto the full timestamp grid
        pos_series  = np.zeros(len(timestamps))
        pnl_series  = np.zeros(len(timestamps))
        last_pos, last_pnl = 0, 0.0
        trade_ts_sorted = sorted(pos_at_ts.keys())
        ti = 0  # pointer into trade timestamps
        for i, ts in enumerate(timestamps):
            while ti < len(trade_ts_sorted) and trade_ts_sorted[ti] <= ts:
                last_pos = pos_at_ts[trade_ts_sorted[ti]]
                last_pnl = pnl_at_ts[trade_ts_sorted[ti]]
                ti += 1
            pos_series[i] = last_pos
            # live mark-to-market
            mid = mid_lookup.get(ts, price_series.iloc[i] if i < len(price_series) else 0)
            pnl_series[i] = last_pnl + (last_pos - (pos_at_ts[trade_ts_sorted[ti-1]] if ti > 0 else 0) + (pos_at_ts[trade_ts_sorted[ti-1]] if ti > 0 else 0)) * 0
            # simpler: cash locked in + current pos MTM
            pnl_series[i] = last_pnl  # pnl_at_ts is already MTM at last trade; drift between trades

        # Recompute clean MTM pnl on full grid
        cash2, pos2 = 0.0, 0
        all_trade_map = {}  # ts → list of (qty, price)
        for _, r in grp.iterrows():
            all_trade_map.setdefault(r["timestamp"], []).append((r["qty"], r["price"]))
        pnl_clean = np.zeros(len(timestamps))
        for i, ts in enumerate(timestamps):
            if ts in all_trade_map:
                for q, p in all_trade_map[ts]:
                    if q > 0:  cash2 -= q * p
                    else:      cash2 += (-q) * p
                    pos2 += q
            pnl_clean[i] = cash2 + pos2 * price_series.iloc[i]

        stats[trader] = dict(
            pos_series  = pos_series,
            pnl_series  = pnl_clean,
            buy_ts      = np.array(buy_ts),
            buy_px      = np.array(buy_px),
            buy_qty     = np.array(buy_qty),
            sell_ts     = np.array(sell_ts),
            sell_px     = np.array(sell_px),
            sell_qty    = np.array(sell_qty),
        )
    return stats


# ── Main loop ─────────────────────────────────────────────────────────────────
for day in days:
    for product in products:
        # ── Price data (same as DP cell) ──────────────────────────────────────
        subset = df_total[
            (df_total["product"] == product) & (df_total["day"] == day)
        ].copy().sort_values("timestamp").reset_index(drop=True)

        if subset.empty:
            print(f"No price data for {product} day {day}")
            continue

        subset["mid_price"] = subset["mid_price"].replace(0, np.nan).ffill()
        prices_arr = subset["mid_price"].to_numpy(dtype=float)
        timestamps = subset["timestamp"].to_numpy()

        # ── Oracle DP ─────────────────────────────────────────────────────────
        trades_list, max_pnl, oracle_pnl = oracle_dp(prices_arr, HALF_SPREAD, POSITION_LIMIT)
        trades_arr  = np.array(trades_list)
        buy_mask    = trades_arr > 0
        sell_mask   = trades_arr < 0

        # ── Bot trades ────────────────────────────────────────────────────────
        bot_df   = load_bot_trades(day, product)
        bot_stats = compute_bot_stats(bot_df, subset["mid_price"], timestamps) if not bot_df.empty else {}
        traders   = sorted(bot_stats.keys())
        colours   = {t: TRADER_PALETTE[i % len(TRADER_PALETTE)] for i, t in enumerate(traders)}

        # ── Figure: 3 rows — price, PnL, position ────────────────────────────
        n_rows = 3
        fig = make_subplots(
            rows=n_rows, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            row_heights=[0.50, 0.28, 0.22],
            subplot_titles=(
                f"Price + Oracle & Bot Trades — {product} (Day {day})",
                f"Cumulative PnL  [oracle ceiling = {max_pnl:,.0f}]",
                "Bot Net Position",
            )
        )

        # ── Row 1: Price path ─────────────────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=timestamps, y=prices_arr,
            mode="lines", name="Mid Price",
            line=dict(color="rgba(160,160,160,0.4)", width=1.2),
            showlegend=False,
        ), row=1, col=1)

        # Oracle buy markers
        if buy_mask.any():
            fig.add_trace(go.Scatter(
                x=timestamps[buy_mask], y=prices_arr[buy_mask],
                mode="markers", name="Oracle Buy",
                marker=dict(symbol="triangle-up", size=10,
                            color="#00c97a", line=dict(color="#007a49", width=1)),
                text=[f"+{d}" for d in trades_arr[buy_mask]],
                hovertemplate="ORACLE BUY %{text}<br>%{y:.1f} @ t=%{x}<extra></extra>",
            ), row=1, col=1)

        # Oracle sell markers
        if sell_mask.any():
            fig.add_trace(go.Scatter(
                x=timestamps[sell_mask], y=prices_arr[sell_mask],
                mode="markers", name="Oracle Sell",
                marker=dict(symbol="triangle-down", size=10,
                            color="#ff4455", line=dict(color="#aa1122", width=1)),
                text=[f"{d}" for d in trades_arr[sell_mask]],
                hovertemplate="ORACLE SELL %{text}<br>%{y:.1f} @ t=%{x}<extra></extra>",
            ), row=1, col=1)

        # Bot trades per trader — use actual TRADE prices (not mid), smaller markers
        for trader in traders:
            col = colours[trader]
            s   = bot_stats[trader]

            if len(s["buy_ts"]):
                sz = np.clip(s["buy_qty"] * 3, 6, 18).astype(float)
                fig.add_trace(go.Scatter(
                    x=s["buy_ts"], y=s["buy_px"],
                    mode="markers", name=f"{trader} Buy",
                    marker=dict(symbol="triangle-up", size=sz,
                                color=col, opacity=0.75,
                                line=dict(color="white", width=0.5)),
                    text=[f"+{q}" for q in s["buy_qty"]],
                    hovertemplate=f"<b>{trader}</b> BUY %{{text}}<br>%{{y:.1f}} @ t=%{{x}}<extra></extra>",
                ), row=1, col=1)

            if len(s["sell_ts"]):
                sz = np.clip(s["sell_qty"] * 3, 6, 18).astype(float)
                fig.add_trace(go.Scatter(
                    x=s["sell_ts"], y=s["sell_px"],
                    mode="markers", name=f"{trader} Sell",
                    marker=dict(symbol="triangle-down", size=sz,
                                color=col, opacity=0.75,
                                line=dict(color="white", width=0.5)),
                    text=[f"-{q}" for q in s["sell_qty"]],
                    hovertemplate=f"<b>{trader}</b> SELL %{{text}}<br>%{{y:.1f}} @ t=%{{x}}<extra></extra>",
                ), row=1, col=1)

        # ── Row 2: PnL curves ─────────────────────────────────────────────────
        # Oracle PnL
        fig.add_trace(go.Scatter(
            x=timestamps, y=oracle_pnl,
            mode="lines", name="Oracle PnL",
            line=dict(color="#4488ff", width=2),
            fill="tozeroy", fillcolor="rgba(68,136,255,0.08)",
        ), row=2, col=1)

        # Bot PnL per trader
        for trader in traders:
            col = colours[trader]
            fig.add_trace(go.Scatter(
                x=timestamps, y=bot_stats[trader]["pnl_series"],
                mode="lines", name=f"{trader} PnL",
                line=dict(color=col, width=1.5, dash="dot"),
            ), row=2, col=1)

        fig.add_hline(y=0, line_dash="dot", line_color="gray", row=2, col=1)

        # ── Row 3: Bot positions ──────────────────────────────────────────────
        for trader in traders:
            col = colours[trader]
            fig.add_trace(go.Scatter(
                x=timestamps, y=bot_stats[trader]["pos_series"],
                mode="lines", name=f"{trader} Pos",
                line=dict(color=col, width=1.3),
                showlegend=False,
            ), row=3, col=1)

        fig.add_hline(y=0, line_dash="dot", line_color="gray", row=3, col=1)

        fig.update_layout(
            height=900,
            template="plotly_white",
            showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.01,
                        xanchor="right", x=1, font=dict(size=10)),
            title_text=(
                f"Oracle vs Bot Comparison — {product} | Day {day} | "
                f"Limit={POSITION_LIMIT} | HalfSpread={HALF_SPREAD}"
            ),
        )
        fig.update_yaxes(title_text="Price",    row=1, col=1)
        fig.update_yaxes(title_text="PnL",      row=2, col=1)
        fig.update_yaxes(title_text="Position", row=3, col=1)
        fig.update_xaxes(title_text="Timestamp", row=3, col=1)

        # ── Console summary ───────────────────────────────────────────────────
        print(f"\n{product} Day {day} — Oracle ceiling: {max_pnl:,.0f}")
        for trader in traders:
            s = bot_stats[trader]
            final_pnl = s["pnl_series"][-1]
            n_b = len(s["buy_ts"]); n_s = len(s["sell_ts"])
            print(f"  {trader:10s}: {n_b:3d} buys  {n_s:3d} sells  final PnL = {final_pnl:>10,.1f}  "
                  f"({100*final_pnl/max_pnl:.1f}% of oracle)")

        fig.show()



HYDROGEL_PACK Day 1 — Oracle ceiling: 293,097
  Mark 14   : 183 buys  187 sells  final PnL =   14,950.0  (5.1% of oracle)
  Mark 22   :   3 buys    2 sells  final PnL =     -193.0  (-0.1% of oracle)
  Mark 38   : 189 buys  186 sells  final PnL =  -14,757.0  (-5.0% of oracle)



HYDROGEL_PACK Day 2 — Oracle ceiling: 269,776
  Mark 14   : 142 buys  161 sells  final PnL =    1,061.0  (0.4% of oracle)
  Mark 22   :   4 buys    4 sells  final PnL =       67.0  (0.0% of oracle)
  Mark 38   : 165 buys  146 sells  final PnL =   -1,128.0  (-0.4% of oracle)



HYDROGEL_PACK Day 3 — Oracle ceiling: 278,348
  Mark 14   : 171 buys  159 sells  final PnL =    8,404.0  (3.0% of oracle)
  Mark 22   :   4 buys    2 sells  final PnL =      103.0  (0.0% of oracle)
  Mark 38   : 161 buys  175 sells  final PnL =   -8,507.0  (-3.1% of oracle)
